In [2]:
# ============================================================
# ВИБРОДИАГНОСТИКА - ПОЛНАЯ ВЕРСИЯ
# Все функции + игра "Угадай дефект" + выбор фильтров
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from ipywidgets import interactive, Dropdown, IntSlider, Button, Output, VBox, HBox
from IPython.display import display, HTML, clear_output
import random
import warnings
warnings.filterwarnings('ignore')

# Настройка matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 12)
plt.rcParams['figure.dpi'] = 100

# ============================================================
# 1. БАЗА ДАННЫХ ПОДШИПНИКОВ
# ============================================================
BEARINGS_DB = {
    '6205':  {'n': 9,  'd': 7.94,  'D': 38.5,  'alpha': 0,  'type': 'Радиальный шариковый'},
    '6208':  {'n': 9,  'd': 12.70, 'D': 57.0,  'alpha': 0,  'type': 'Радиальный шариковый'},
    '6305':  {'n': 8,  'd': 11.00, 'D': 43.0,  'alpha': 0,  'type': 'Радиальный шариковый'},
    '6308':  {'n': 8,  'd': 15.08, 'D': 65.0,  'alpha': 0,  'type': 'Радиальный шариковый'},
    '6310':  {'n': 8,  'd': 18.00, 'D': 80.0,  'alpha': 0,  'type': 'Радиальный шариковый'},
    '6312':  {'n': 8,  'd': 22.00, 'D': 95.0,  'alpha': 0,  'type': 'Радиальный шариковый'},
    '22210': {'n': 28, 'd': 16.0,  'D': 68.0,  'alpha': 0,  'type': 'Сферический роликовый'},
    '22220': {'n': 28, 'd': 32.0,  'D': 138.0, 'alpha': 0,  'type': 'Сферический роликовый'},
    '30210': {'n': 18, 'd': 17.0,  'D': 68.0,  'alpha': 15, 'type': 'Конический роликовый'},
    '7014':  {'n': 20, 'd': 11.0,  'D': 90.0,  'alpha': 15, 'type': 'Угловой контактный'},
}

# ============================================================
# 2. РАСЧЕТ ЧАСТОТ
# ============================================================
def calculate_frequencies(bearing_name, rpm):
    b = BEARINGS_DB[bearing_name]
    fr = rpm / 60.0
    d_D = b['d'] / b['D']
    cos_a = np.cos(np.radians(b['alpha']))

    bpfo = (b['n'] / 2) * fr * (1 - d_D * cos_a)
    bpfi = (b['n'] / 2) * fr * (1 + d_D * cos_a)
    bsf = (b['D'] / (2 * b['d'])) * fr * (1 - (d_D * cos_a)**2)
    ftf = (fr / 2) * (1 - d_D * cos_a)

    return {'fr': fr, 'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf, '2xBSF': 2 * bsf}

# ============================================================
# 3. ГЕНЕРАТОР СИГНАЛА
# ============================================================
def generate_fault_signal(rpm, bearing_name, defect_type, stage, duration=0.5, fs=50000, resonance_freq=2500):
    t = np.arange(0, duration, 1/fs)
    freqs = calculate_frequencies(bearing_name, rpm)
    fr = freqs['fr']

    noise_level = 0.02
    signal_data = np.random.normal(0, noise_level, len(t))

    if defect_type == 'none':
        return t, signal_data, freqs

    if defect_type == 'BPFO':
        fault_freq, mod_freq = freqs['BPFO'], None
    elif defect_type == 'BPFI':
        fault_freq, mod_freq = freqs['BPFI'], fr
    elif defect_type == 'BSF':
        fault_freq, mod_freq = freqs['2xBSF'], 2 * freqs['FTF']
    elif defect_type == 'FTF':
        fault_freq, mod_freq = freqs['FTF'], fr
    else:
        return t, signal_data, freqs

    amp_dict = {1: 0.15, 2: 0.4, 3: 0.8, 4: 1.5}
    amplitude = amp_dict.get(stage, 0.5)

    mod_dict = {1: 0.1, 2: 0.3, 3: 0.6, 4: 0.85}
    mod_depth = mod_dict.get(stage, 0.3) if mod_freq else 0

    harm_dict = {1: 1, 2: 2, 3: 3, 4: 5}
    n_harmonics = harm_dict.get(stage, 2)

    for h in range(1, n_harmonics + 1):
        current_freq = fault_freq * h
        current_amp = amplitude / h
        impulse_period = 1.0 / current_freq
        impulse_times = np.arange(0, duration, impulse_period)

        for imp_time in impulse_times:
            modulation = (1 + mod_depth * np.sin(2 * np.pi * mod_freq * imp_time)) if mod_freq else 1.0

            impulse_duration = 0.003
            n_samples = int(impulse_duration * fs)
            impulse_t = np.arange(n_samples) / fs

            resonance = current_amp * modulation * np.sin(2 * np.pi * resonance_freq * impulse_t)
            decay = np.exp(-800 * impulse_t)
            impulse = resonance * decay

            start_idx = int(imp_time * fs)
            end_idx = start_idx + n_samples

            if end_idx < len(signal_data):
                signal_data[start_idx:end_idx] += impulse

    return t, signal_data, freqs

# ============================================================
# 4. ОБРАБОТКА (ОГИБАЮЩАЯ) - С ВЫБОРОМ ФИЛЬТРА
# ============================================================
def process_envelope(sig, fs, fc=2500, bandwidth='1/3_octave'):
    """
    Обработка сигнала с выбором типа фильтра
    bandwidth: '1/3_octave' или '1_octave'
    """
    if bandwidth == '1/3_octave':
        f_low, f_high = fc / 1.122, fc * 1.122
        filter_desc = '1/3 октавы'
    else:  # 1_octave
        f_low, f_high = fc / 1.414, fc * 1.414
        filter_desc = '1 октава'

    nyquist = fs / 2
    if f_high >= nyquist:
        f_high = nyquist * 0.9

    sos = signal.butter(4, [f_low, f_high], btype='band', fs=fs, output='sos')
    filtered = signal.sosfilt(sos, sig)

    rectified = np.abs(filtered)
    cutoff = min(1500, fs / 4)
    sos_lp = signal.butter(4, cutoff, btype='low', fs=fs, output='sos')
    envelope = signal.sosfilt(sos_lp, rectified)

    N = len(envelope)
    yf = fft(envelope)
    xf = fftfreq(N, 1/fs)[:N//2]
    amplitude = 2.0/N * np.abs(yf[:N//2])

    mask = xf <= 1000
    return xf[mask], amplitude[mask], envelope, filter_desc

# ============================================================
# 5. ВИЗУАЛИЗАЦИЯ (С ФИЛЬТРОМ И FC)
# ============================================================
def plot_results(bearing_name, rpm, defect_type, stage, fc, bandwidth):
    """Основная функция визуализации с выбором фильтра"""

    # Генерация сигнала
    t, sig, freqs = generate_fault_signal(
        rpm=rpm,
        bearing_name=bearing_name,
        defect_type=defect_type,
        stage=stage,
        duration=0.3,
        fs=50000,
        resonance_freq=fc
    )

    # Обработка с выбранным фильтром
    xf_env, amp_env, envelope, filter_desc = process_envelope(sig, fs=50000, fc=fc, bandwidth=bandwidth)

    # === СОЗДАНИЕ ФИГУРЫ ===
    fig = plt.figure(figsize=(14, 12))

    title_text = (f"Подшипник {bearing_name} | {BEARINGS_DB[bearing_name]['type']} | "
                 f"{rpm} об/мин | Дефект: {defect_type} | Стадия: {stage} | "
                 f"fc={fc} Гц | Фильтр: {filter_desc}")
    fig.suptitle(title_text, fontsize=13, fontweight='bold', y=0.98)

    # === ГРАФИК 1: ВРЕМЕННОЙ СИГНАЛ ===
    ax1 = plt.subplot(4, 1, 1)
    ax1.plot(t * 1000, sig, linewidth=0.3, color='steelblue')
    ax1.set_xlabel('Время, мс')
    ax1.set_ylabel('Ускорение, g')
    ax1.set_title('Исходный временной сигнал (сырой)')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 50)

    # === ГРАФИК 2: ОГИБАЮЩАЯ ===
    ax2 = plt.subplot(4, 1, 2)
    ax2.plot(t * 1000, envelope, linewidth=0.5, color='red')
    ax2.set_xlabel('Время, мс')
    ax2.set_ylabel('Огибающая, g')
    ax2.set_title(f'Огибающая (fc={fc} Гц, фильтр {filter_desc})')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 50)

    # === ГРАФИК 3: СПЕКТР ОГИБАЮЩЕЙ (ЛИНЕЙНЫЙ) ===
    ax3 = plt.subplot(4, 1, 3)
    ax3.plot(xf_env, amp_env, linewidth=0.8, color='darkgreen')
    ax3.set_xlabel('Частота, Гц')
    ax3.set_ylabel('Амплитуда, g')
    ax3.set_title('Спектр огибающей (линейная шкала)')
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(0, 500)

    # === ГРАФИК 4: СПЕКТР ОГИБАЮЩЕЙ (ЛОГАРИФМИЧЕСКИЙ) ===
    ax4 = plt.subplot(4, 1, 4)
    amp_db = 20 * np.log10(amp_env + 1e-10)
    ax4.plot(xf_env, amp_db, linewidth=0.8, color='purple')
    ax4.set_xlabel('Частота, Гц')
    ax4.set_ylabel('Амплитуда, дБ')
    ax4.set_title('Спектр огибающей (логарифмическая шкала — видны боковые полосы!)')
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(0, 500)
    ax4.set_ylim(-100, np.max(amp_db) + 10)

    # === ОТМЕТКА РАСЧЕТНЫХ ЧАСТОТ ===
    colors = {
        'BPFO': 'red',
        'BPFI': 'blue',
        'BSF': 'green',
        'FTF': 'orange',
        '2xBSF': 'magenta'
    }

    for ax in [ax3, ax4]:
        for defect_name, color in colors.items():
            if defect_name in freqs:
                f = freqs[defect_name]
                if f < 500:
                    ax.axvline(x=f, color=color, linestyle='--',
                              alpha=0.6, linewidth=1.5, label=f'{defect_name}={f:.1f} Гц')
                    for h in [2, 3]:
                        if f * h < 500:
                            ax.axvline(x=f*h, color=color, linestyle=':',
                                      alpha=0.4, linewidth=1)

    ax3.legend(loc='upper right', fontsize=8, ncol=2)

    # === ИНФОРМАЦИОННАЯ ПАНЕЛЬ ===
    info_text = "РАСЧЕТНЫЕ ЧАСТОТЫ:\n"
    info_text += f"   BPFO: {freqs['BPFO']:.2f} Гц | BPFI: {freqs['BPFI']:.2f} Гц\n"
    info_text += f"   BSF: {freqs['BSF']:.2f} Гц | FTF: {freqs['FTF']:.2f} Гц | 2*BSF: {freqs['2xBSF']:.2f} Гц\n"
    info_text += f"НАСТРОЙКИ: fc={fc} Гц | Фильтр: {filter_desc} | Обороты: {freqs['fr']:.2f} Гц ({rpm} об/мин)"

    fig.text(0.02, 0.01, info_text, fontsize=9, family='monospace',
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout(rect=[0, 0.15, 1, 0.96])
    plt.show()

# ============================================================
# 6. ЗАГОЛОВОК
# ============================================================
display(HTML("""
<div style='background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
            padding: 20px; border-radius: 10px; color: white; margin-bottom: 20px;'>
    <h2 style='margin:0;'>🔧 Виртуальный стенд вибродиагностики</h2>
    <p style='margin:5px 0 0 0;'>Генератор спектров огибающей с дефектами подшипников</p>
</div>
"""))

# ============================================================
# 7. ИНТЕРАКТИВНЫЙ ИНТЕРФЕЙС (С ФИЛЬТРОМ И FC)
# ============================================================
interactive_plot = interactive(
    plot_results,
    bearing_name=Dropdown(
        options=list(BEARINGS_DB.keys()),
        value='6308',
        description='Подшипник:',
        style={'description_width': 'initial'}
    ),
    rpm=IntSlider(
        value=1500, min=300, max=3600, step=50,
        description='Обороты (об/мин):',
        style={'description_width': 'initial'}
    ),
    defect_type=Dropdown(
        options=['none', 'BPFO', 'BPFI', 'BSF', 'FTF'],
        value='BPFO',
        description='Тип дефекта:',
        style={'description_width': 'initial'}
    ),
    stage=IntSlider(
        value=2, min=1, max=4, step=1,
        description='Стадия дефекта (1-4):',
        style={'description_width': 'initial'}
    ),
    fc=Dropdown(
        options=[800, 1600, 2500, 4000, 5000, 6300],
        value=2500,
        description='Центральная частота fc (Гц):',
        style={'description_width': 'initial'}
    ),
    bandwidth=Dropdown(
        options=['1/3_octave', '1_octave'],
        value='1/3_octave',
        description='Тип фильтра:',
        style={'description_width': 'initial'}
    )
)

display(interactive_plot)

# ============================================================
# 8. ШПАРГАЛКА
# ============================================================
display(HTML("""
<div style='background: #f0f8ff; padding: 15px; border-left: 5px solid #4682b4;
            border-radius: 5px; margin-top: 20px;'>
    <h3 style='margin-top:0; color: #2c3e50;'>📚 Шпаргалка для диагностики:</h3>
    <table style='width:100%; border-collapse: collapse;'>
        <tr style='background: #e8f4f8;'>
            <td style='padding: 8px; border: 1px solid #ddd;'><b>BPFO</b></td>
            <td style='padding: 8px; border: 1px solid #ddd;'>
                Пик на частоте BPFO + гармоники. <b>НЕТ боковых полос</b> (кольцо неподвижно)
            </td>
        </tr>
        <tr>
            <td style='padding: 8px; border: 1px solid #ddd;'><b>BPFI</b></td>
            <td style='padding: 8px; border: 1px solid #ddd;'>
                Пик на BPFI + <b>боковые полосы с шагом 1× оборотов</b> (кольцо вращается!)
            </td>
        </tr>
        <tr style='background: #e8f4f8;'>
            <td style='padding: 8px; border: 1px solid #ddd;'><b>BSF (2×BSF)</b></td>
            <td style='padding: 8px; border: 1px solid #ddd;'>
                Пик на 2×BSF + боковые полосы с шагом <b>2×FTF</b>
            </td>
        </tr>
        <tr>
            <td style='padding: 8px; border: 1px solid #ddd;'><b>FTF</b></td>
            <td style='padding: 8px; border: 1px solid #ddd;'>
                Пик на очень низкой частоте (~0.4× оборотов) + гармоники
            </td>
        </tr>
    </table>
    <p style='margin-top: 10px;'>
        <b>💡 Совет:</b> Переключайте шкалу между линейной и логарифмической —
        боковые полосы видны ТОЛЬКО в дБ!
    </p>
    <p style='margin-top: 5px;'>
        <b>🔧 Фильтры:</b> 1/3 октавы — стандарт (селективность), 1 октава — для слабых сигналов
    </p>
</div>
"""))

# ============================================================
# 9. ИГРА "УГАДАЙ ДЕФЕКТ"
# ============================================================
display(HTML('<h3 style="margin-top: 30px;">🎓 Игра "Угадай дефект":</h3>'))
display(HTML('<p>Нажмите "Новый вопрос", изучите спектр, затем нажмите "Показать ответ"</p>'))

# Глобальные переменные для экзамена
exam_data = {}
exam_output = Output()
answer_output = Output()

# Словарь ответов
ANSWERS_DB = {
    'BPFO': "📊 Частота: {BPFO:.2f} Гц\n🔍 Признак: пики на BPFO и её гармониках, НЕТ боковых полос (кольцо неподвижно)",
    'BPFI': "📊 Частота: {BPFI:.2f} Гц\n🔍 Признак: пик на BPFI + боковые полосы с шагом {fr:.1f} Гц (1× оборотов)",
    'BSF': "📊 Частота: 2×BSF = {BSF2:.2f} Гц (НЕ BSF!)\n🔍 Признак: пик на 2×BSF + боковые полосы с шагом {FTF2:.2f} Гц (2×FTF)",
    'FTF': "📊 Частота: {FTF:.2f} Гц (очень низкая!)\n🔍 Признак: пик на низкой частоте ≈ 0.4× оборотов (дефект сепаратора)"
}

STAGE_DESC = {
    1: "только основная частота, гармоник нет",
    2: "появилась 2-я гармоника",
    3: "есть 2-я и 3-я гармоники, чёткие боковые полосы",
    4: "много гармоник — критический дефект!"
}

def generate_exam_question(b):
    """Генерация экзаменационного вопроса"""
    global exam_data

    bearing = random.choice(list(BEARINGS_DB.keys()))
    rpm = random.choice([750, 1000, 1500, 2000, 3000])
    defect = random.choice(['BPFO', 'BPFI', 'BSF', 'FTF'])
    stage = random.randint(1, 4)
    fc = random.choice([1600, 2500, 4000])
    bandwidth = random.choice(['1/3_octave', '1_octave'])

    exam_data = {
        'bearing': bearing,
        'rpm': rpm,
        'defect': defect,
        'stage': stage,
        'fc': fc,
        'bandwidth': bandwidth,
        'freqs': calculate_frequencies(bearing, rpm)
    }

    # Генерация графика
    t, sig, freqs = generate_fault_signal(rpm, bearing, defect, stage, duration=0.3, fs=50000, resonance_freq=fc)
    xf_env, amp_env, envelope, filter_desc = process_envelope(sig, fs=50000, fc=fc, bandwidth=bandwidth)

    with exam_output:
        exam_output.clear_output(wait=True)

        print("=" * 70)
        print("🎲 ЭКЗАМЕНАЦИОННЫЙ ВОПРОС")
        print("=" * 70)
        print(f"⚙️ Подшипник:      {bearing} ({BEARINGS_DB[bearing]['type']})")
        print(f"🔄 Обороты:        {rpm} об/мин")
        print(f"📊 Центральная частота: {fc} Гц")
        print(f"🔧 Фильтр:         {filter_desc}")
        print(f"🔍 Стадия дефекта:  {stage} из 4")
        print("=" * 70)
        print("\n👆 Изучите спектры ниже и определите тип дефекта!")
        print("💡 Подсказка: обратите внимание на наличие/отсутствие боковых полос\n")

        # График
        fig = plt.figure(figsize=(14, 8))

        ax1 = plt.subplot(2, 1, 1)
        ax1.plot(xf_env, amp_env, linewidth=0.8, color='darkgreen')
        ax1.set_title('Спектр огибающей (линейная шкала) — ОПРЕДЕЛИТЕ ДЕФЕКТ')
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim(0, 500)
        ax1.set_ylabel('Амплитуда, g')

        ax2 = plt.subplot(2, 1, 2)
        amp_db = 20 * np.log10(amp_env + 1e-10)
        ax2.plot(xf_env, amp_db, linewidth=0.8, color='purple')
        ax2.set_title('Спектр огибающей (логарифмическая шкала) — ИЩИТЕ БОКОВЫЕ ПОЛОСЫ!')
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim(0, 500)
        ax2.set_ylim(-100, np.max(amp_db) + 10)
        ax2.set_ylabel('Амплитуда, дБ')

        plt.tight_layout()
        plt.show()

        print("\n" + "=" * 70)
        print("📝 Ваш ответ: _________________")
        print("=" * 70)
        print("\n👇 Нажмите кнопку 'Показать ответ' ниже")

def show_exam_answer(b):
    """Показ правильного ответа"""
    if not exam_data:
        with answer_output:
            answer_output.clear_output()
            print("⚠️ Сначала нажмите 'Новый вопрос'!")
        return

    d = exam_data['defect']
    f = exam_data['freqs']
    s = exam_data['stage']

    ans_text = ANSWERS_DB[d].format(
        BPFO=f['BPFO'], BPFI=f['BPFI'],
        BSF2=f['2xBSF'], FTF2=f['FTF']*2,
        fr=f['fr'], FTF=f['FTF']
    )

    with answer_output:
        answer_output.clear_output()
        print("=" * 70)
        print(f"✅ ПРАВИЛЬНЫЙ ОТВЕТ: {d}")
        print("=" * 70)
        print(ans_text)
        print(f"📈 Стадия {s}: {STAGE_DESC[s]}")
        print("=" * 70)

# Кнопки экзамена
btn_new_exam = Button(
    description='🎲 Новый вопрос',
    button_style='primary',
    layout={'width': '200px', 'height': '50px'}
)
btn_new_exam.on_click(generate_exam_question)

btn_show_answer = Button(
    description='👁️ Показать ответ',
    button_style='success',
    layout={'width': '200px', 'height': '50px'}
)
btn_show_answer.on_click(show_exam_answer)

# Отображение кнопок и вывода
display(VBox([
    HBox([btn_new_exam, btn_show_answer]),
    exam_output,
    answer_output
]))

# Автозапуск первого вопроса
generate_exam_question(None)

print("\n✅ Все готово! Ползунки и игра работают.")

interactive(children=(Dropdown(description='Подшипник:', index=3, options=('6205', '6208', '6305', '6308', '63…

BPFO,Пик на частоте BPFO + гармоники. НЕТ боковых полос (кольцо неподвижно)
BPFI,Пик на BPFI + боковые полосы с шагом 1× оборотов (кольцо вращается!)
BSF (2×BSF),Пик на 2×BSF + боковые полосы с шагом 2×FTF
FTF,Пик на очень низкой частоте (~0.4× оборотов) + гармоники



✅ Все готово! Ползунки и игра работают.
